In [1]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

from extracao_comum import ExtracaoVaga, criar_cliente, extrair_vaga

PROCESSED_DIR = Path("../data/processed")
vagas = pd.read_parquet(PROCESSED_DIR / "vagas_limpas.parquet")
print(f"{len(vagas)} vagas carregadas")

283 vagas carregadas


In [2]:
for nome, campo in ExtracaoVaga.model_fields.items():
    tipo = getattr(campo.annotation, "__name__", str(campo.annotation))
    print(f"{nome} ({tipo})\n  {campo.description}\n")

skills_tecnicas (list)
  Tecnologias, linguagens, frameworks e plataformas exigidas ou desejadas, normalizadas em inglês minúsculo (ex.: python, machine learning, aws, docker)

praticas_genai (list)
  Práticas de IA generativa citadas na vaga, em minúsculo (ex.: llm, rag, prompt engineering, ai agents, fine-tuning); vazio se nenhuma

ferramentas_ia_codigo (list)
  Ferramentas de IA para apoiar a escrita de código citadas (ex.: github copilot, cursor, claude code, windsurf); vazio se nenhuma

usa_ia_no_desenvolvimento (Literal)
  Se a vaga espera que a pessoa use IA no próprio fluxo de trabalho de desenvolvimento: 'exige' (requisito), 'valoriza' (diferencial), 'menciona' (cita sem exigir) ou 'nao_menciona'

senioridade (Literal)
  Senioridade indicada no título ou na descrição; 'lider' cobre tech lead, coordenação e gestão

modalidade (Literal)
  Modalidade de trabalho declarada na descrição

exige_ingles (bool)
  True se a vaga exige inglês ou se a descrição inteira é em inglês para em

In [3]:
cliente = criar_cliente()
MODELO = "openai/gpt-oss-120b"

In [4]:
teste = extrair_vaga(cliente, MODELO, vagas.iloc[0]["descricao"])
print(f"=== {vagas.iloc[0]['job_title']} — {vagas.iloc[0]['company']} ===")
teste

=== AI Builders — Xertica.ai ===


ExtracaoVaga(skills_tecnicas=[], praticas_genai=['ai agents', 'copilots', 'automation', 'llm'], ferramentas_ia_codigo=[], usa_ia_no_desenvolvimento='exige', senioridade='nao_informado', modalidade='nao_informado', exige_ingles=False, exige_formacao_superior=False)

In [5]:
db = sqlite3.connect(PROCESSED_DIR / "extracao.sqlite")
db.execute("""
    CREATE TABLE IF NOT EXISTS extracoes (
        url_base TEXT,
        modelo TEXT,
        json TEXT,
        extraido_em TEXT DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (url_base, modelo)
    )
""")

feitas = {url for (url,) in db.execute(
    "SELECT url_base FROM extracoes WHERE modelo = ?", (MODELO,))}
pendentes = vagas[~vagas["url_base"].isin(feitas)]
print(f"{len(feitas)} vagas já extraídas, {len(pendentes)} pendentes")

283 vagas já extraídas, 0 pendentes


In [6]:
from tqdm.auto import tqdm

for _, vaga in tqdm(pendentes.iterrows(), total=len(pendentes)):
    extracao = extrair_vaga(cliente, MODELO, vaga["descricao"])
    db.execute(
        "INSERT INTO extracoes (url_base, modelo, json) VALUES (?, ?, ?)",
        (vaga["url_base"], MODELO, extracao.model_dump_json()),
    )
    db.commit()

total = db.execute(
    "SELECT COUNT(*) FROM extracoes WHERE modelo = ?", (MODELO,)).fetchone()[0]
print(f"total extraído pelo modelo principal: {total}")

0it [00:00, ?it/s]

total extraído pelo modelo principal: 283


In [7]:
extraidas = pd.read_sql(
    "SELECT url_base, json FROM extracoes WHERE modelo = ?", db, params=(MODELO,))
campos = pd.json_normalize(extraidas["json"].map(json.loads))
resultado = vagas.merge(pd.concat([extraidas[["url_base"]], campos], axis=1), on="url_base")

resultado.to_parquet(PROCESSED_DIR / "vagas_extraidas.parquet", index=False)
print(f"{len(resultado)} vagas salvas em {PROCESSED_DIR / 'vagas_extraidas.parquet'}")

283 vagas salvas em ..\data\processed\vagas_extraidas.parquet


In [8]:
for coluna in ["senioridade", "modalidade", "usa_ia_no_desenvolvimento"]:
    print(resultado[coluna].value_counts().to_string(), "\n")
print(f"exige inglês: {resultado['exige_ingles'].mean():.0%} · "
      f"exige formação superior: {resultado['exige_formacao_superior'].mean():.0%}\n")
resultado["skills_tecnicas"].explode().value_counts().head(20)

senioridade
nao_informado    209
senior            51
pleno             19
junior             2
estagio            1
lider              1 

modalidade
nao_informado    147
remoto            74
hibrido           36
presencial        26 

usa_ia_no_desenvolvimento
exige           256
nao_menciona     21
valoriza          6 

exige inglês: 39% · exige formação superior: 36%



skills_tecnicas
python              217
machine learning     97
aws                  97
sql                  79
azure                63
ci/cd                63
langchain            62
docker               57
pytorch              54
tensorflow           52
mlops                51
gcp                  49
git                  44
langgraph            43
scikit-learn         38
api                  37
kubernetes           36
nlp                  36
deep learning        33
crewai               31
Name: count, dtype: int64